In [ ]:
import pandas as pd
import numpy as np
import glob
import os

In [ ]:
DATA_PATH = "../data/raw"

all_files = sorted(glob.glob(os.path.join(DATA_PATH, "*.csv")))

dfs = []

for file in all_files:
    df_temp = pd.read_csv(file)

    season = os.path.basename(file).replace("pbp", "").replace(".csv", "")
    df_temp["season"] = int(season)

    dfs.append(df_temp)

df = pd.concat(dfs, ignore_index=True)

print("Shape:", df.shape)
print("Games:", df["gameid"].nunique())
df.head()

In [ ]:
df = df.sort_values(["gameid", "period"])
df = df.reset_index(drop=True)

In [ ]:
df["h_pts"] = df.groupby("gameid")["h_pts"].ffill()
df["a_pts"] = df.groupby("gameid")["a_pts"].ffill()

In [ ]:
df["score_diff"] = df["h_pts"] - df["a_pts"]

In [ ]:
def convert_clock(clock):
    clock = clock.replace("PT", "")
    minutes = clock.split("M")[0]
    seconds = clock.split("M")[1].replace("S", "")
    return int(minutes) * 60 + float(seconds)

df["time_remaining"] = df["clock"].apply(convert_clock)

In [ ]:
df["score_diff_change"] = df.groupby("gameid")["score_diff"].diff().fillna(0)

In [ ]:
df["momentum"] = (
    df.groupby("gameid")["score_diff_change"]
    .transform(lambda x: x.rolling(window=10, min_periods=1).mean())
    .fillna(0)
)

In [ ]:
final_scores = df.groupby("gameid").tail(1)

game_results = final_scores[["gameid", "h_pts", "a_pts"]].copy()

game_results["home_win"] = (game_results["h_pts"] > game_results["a_pts"]).astype(int)

In [ ]:
df = df.merge(
    game_results[["gameid", "home_win"]],
    on="gameid",
    how="left"
)

In [ ]:
ml_df = df[[
    "gameid",
    "season",
    "period",
    "time_remaining",
    "score_diff",
    "momentum",
    "home_win"
]].copy()

print(ml_df.shape)
ml_df.head()

In [ ]:
ml_df["home_win"].value_counts()

In [ ]:
pip install pyarrow

In [ ]:
ml_df.to_parquet("../data/processed/ml_dataset.parquet", index=False)